# 06-00 AdaGrad：从 SGD 到自适应学习率

这一节专门把 AdaGrad 讲慢一点。

前面你说得对：如果一上来就抛公式，初学者会感觉这些符号像凭空出现。我们这节按更自然的顺序来：

```text
先看 SGD 遇到了什么问题
再看 AdaGrad 想解决什么
最后才看公式为什么长这样
```

AdaGrad 的核心不是某个复杂公式，而是一句话：

```text
不同参数，不一定应该用同样大的步子。
```

## 1. 先回到 SGD

SGD 更新参数的公式是：

$$
\theta_t=\theta_{t-1}-\eta g_t
$$

这里有三个东西：

- $\theta_{t-1}$：更新前的参数
- $g_t$：当前梯度
- $\eta$：学习率，也就是基础步长

这句话翻译成人话就是：

```text
根据当前梯度，把参数往能让损失下降的方向挪一步。
```

问题在于：SGD 里所有参数共用同一个 $\eta$。

也就是说，不管某个参数经常被更新，还是很少被更新，它们都听同一个步长命令。

## 2. 为什么统一学习率有时候不合适

想象模型里有两个参数：

```text
参数 A：几乎每批数据都会影响它，梯度经常很大
参数 B：只有少数样本会影响它，梯度不常出现
```

如果它们都用同一个学习率，会有两个问题。

对参数 A 来说，它本来就经常被推动，如果步子还很大，就容易左右摇摆，训练不稳定。

对参数 B 来说，它本来更新机会就少，如果步子还被统一压得很小，就可能学得特别慢。

所以我们希望优化器能更细致一点：

```text
每个参数根据自己的历史情况，自动决定实际走多大。
```

这就是 AdaGrad 出现的动机。

## 3. AdaGrad 的直觉

AdaGrad 做了一件很朴素的事：给每个参数单独记账。

它关心的问题是：

```text
这个参数过去的梯度大不大？
```

如果过去经常大，说明这个参数已经被强烈影响过很多次，以后应该谨慎一点。

如果过去不怎么大，说明这个参数没有被强烈更新过，就不要太早把它压小。

所以 AdaGrad 的策略是：

```text
历史梯度越大，后面的实际步子越小。
```

注意，这里说的不是把全局学习率 $\eta$ 改掉，而是给每个参数算一个自己的“实际步子”。

## 4. 第一个公式：记账

设第 $t$ 次更新时，某个参数的梯度是：

$$
g_t
$$

AdaGrad 先把这个梯度平方：

$$
g_t^2
$$

为什么平方？

因为梯度有正负，正负表示方向；但 AdaGrad 现在只想统计“大小”。平方以后，不管原来是正还是负，都变成正数。

然后把它加入历史账本：

$$
s_t=s_{t-1}+g_t^2
$$

这个式子可以读成：

```text
新的历史账本 = 旧的历史账本 + 这一次梯度大小的记录
```

所以 $s_t$ 不是神秘符号，它就是这个参数到目前为止的“梯度活跃度记录”。

## 5. 第二个公式：用账本调整步子

SGD 的原始更新是：

$$
\theta_t=\theta_{t-1}-\eta g_t
$$

AdaGrad 改成：

$$
\theta_t=\theta_{t-1}-\eta\frac{g_t}{\sqrt{s_t}+\epsilon}
$$

你可以把它拆成两层。

第一层没有变：

$$
\theta_t=\theta_{t-1}-\text{更新量}
$$

第二层是更新量变了：

$$
\text{更新量}=\eta\frac{g_t}{\sqrt{s_t}+\epsilon}
$$

关键就在分母：

$$
\sqrt{s_t}+\epsilon
$$

如果 $s_t$ 很大，分母就大，更新量就小。

如果 $s_t$ 不大，分母就没那么大，更新量相对更大。

所以这个分母的作用就是“刹车”。

```text
历史梯度账越大，刹车越重。
```

## 6. 为什么要开根号

你可能会问：既然 $s_t$ 是梯度平方的累加，为什么更新时用 $\sqrt{s_t}$，而不是直接用 $s_t$？

直观理解就够了：

$s_t$ 里面加的是 $g_t^2$，它的量级被平方放大了。

如果直接拿 $s_t$ 放到分母里，刹车可能太猛。

开根号以后：

$$
\sqrt{s_t}
$$

它大致回到和梯度 $g_t$ 更接近的量级，调整会更自然。

所以你可以这样记：

```text
平方是为了统计大小
开根号是为了把尺度拉回来
```

## 7. 为什么要加 epsilon

公式里还有一个很小的数：

$$
\epsilon
$$

完整分母是：

$$
\sqrt{s_t}+\epsilon
$$

它主要不是算法思想，而是计算保护。

如果一开始 $s_t=0$，分母可能变成 $0$。

除以 $0$ 是不允许的，所以加一个很小很小的 $\epsilon$，比如：

$$
10^{-8}
$$

这样可以让计算更稳定。

一句话记：

```text
epsilon 是安全垫，防止除以 0。
```

## 8. 用一个小数字看懂 AdaGrad

假设基础学习率是：

$$
\eta=0.1
$$

某个参数第一次梯度是：

$$
g_1=2
$$

历史账本为：

$$
s_1=0+2^2=4
$$

忽略 $\epsilon$，实际更新量大约是：

$$
0.1\times\frac{2}{\sqrt{4}}=0.1\times\frac{2}{2}=0.1
$$

如果后面这个参数一直梯度很大，$s_t$ 会不断变大。

假设某一刻：

$$
s_t=100
$$

同样梯度 $g_t=2$，更新量变成：

$$
0.1\times\frac{2}{\sqrt{100}}=0.1\times\frac{2}{10}=0.02
$$

你看，同样是梯度 $2$，因为历史账本变大了，实际步子从 $0.1$ 变成了 $0.02$。

这就是 AdaGrad 的“自适应”。

## 9. AdaGrad 的优点

AdaGrad 的优点是：它比 SGD 更照顾不同参数的差异。

特别是对稀疏特征有帮助。

什么叫稀疏特征？

比如文本任务里，有些词很常见，有些词很少出现。

常见词对应的参数经常被更新，历史梯度账很快变大，AdaGrad 会让它后面走得谨慎一点。

少见词对应的参数不常被更新，历史梯度账没那么大，它一旦出现，就不会被压得太死。

所以 AdaGrad 常被描述为：

```text
对频繁出现的参数小步走，对不频繁出现的参数相对大步走。
```

## 10. AdaGrad 的缺点

AdaGrad 最大的问题是：账本只加不减。

$$
s_t=s_{t-1}+g_t^2
$$

这个式子没有遗忘机制。

训练越久，$s_t$ 越大。

分母越大：

$$
\sqrt{s_t}+\epsilon
$$

实际更新越小：

$$
\eta\frac{g_t}{\sqrt{s_t}+\epsilon}
$$

所以训练后期可能出现：

```text
模型还需要继续学
但步子已经被压得太小
越走越慢
```

这就是 AdaGrad 为什么后来会被 RMSProp 改进。

## 11. AdaGrad 到 RMSProp 的过渡

如果 AdaGrad 的问题是“太记历史”，那自然的改法就是：

```text
不要永久记住所有历史，只更关注最近的梯度情况。
```

于是 RMSProp 把 AdaGrad 的：

$$
s_t=s_{t-1}+g_t^2
$$

改成指数移动平均：

$$
s_t=\beta s_{t-1}+(1-\beta)g_t^2
$$

这一步你现在应该能看懂了。

AdaGrad 是一直累加；RMSProp 是旧账打折，新账加入。

所以 RMSProp 可以缓解 AdaGrad 后期学习率下降太快的问题。

## 12. 本节总结

AdaGrad 的逻辑链是：

```text
SGD 所有参数共用一个学习率
-> 不同参数的更新频率和梯度大小不同
-> 希望每个参数有自己的实际步子
-> AdaGrad 给每个参数累计历史梯度平方
-> 历史梯度越大，分母越大，实际步子越小
-> 优点：能自适应调整参数步长，适合稀疏特征
-> 缺点：历史账本只加不减，后期学习率可能太小
-> RMSProp 用指数移动平均来改进它
```

先记住一句话：

```text
AdaGrad = 给每个参数记梯度账，账越大，以后走得越小心。
```